# STEP 1 — Silver Metadata Table

In [0]:
# %sql
# CREATE TABLE IF NOT EXISTS ottconsumption.silver.metadata_config (
#     source_table STRING,
#     target_table STRING,
#     primary_key STRING,
#     watermark_column STRING,
#     load_type STRING
# )
# USING DELTA;

In [0]:
# %sql
# INSERT INTO ott_consumption.silver.metadata_config VALUES
# ('bronze.users', 'silver.users', 'user_id', 'ingestion_time', 'SCD2'),
# ('bronze.subscriptions', 'silver.subscriptions', 'subscription_id', 'ingestion_time', 'SCD2'),
# ('bronze.payments', 'silver.payments', 'payment_id', 'ingestion_time', 'incremental'),
# ('bronze.watch_history', 'silver.watch_history', 'watch_id', 'ingestion_time', 'incremental');

# STEP 2 — Silver Watermark Table

In [0]:
%sql

CREATE TABLE IF NOT EXISTS ottconsumption.silver.watermark_table (
    table_name STRING,
    last_processed_value TIMESTAMP
)
USING DELTA;

# Step 3 — Read Bronze Incrementally

In [0]:
df = spark.table("ottconsumption.bronze.users")

wm = spark.sql("""
SELECT last_processed_value 
FROM ottconsumption.silver.watermark_table 
WHERE table_name = 'users_dim'
""")

if wm.count() > 0:
    last_wm = wm.collect()[0][0]
    df = df.filter(f"ingestion_time > '{last_wm}'")

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import *

df = df.filter(col("user_id").isNotNull())
df = df.filter(col("age") > 0)
# df = df.dropDuplicates(["user_id"])

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import *
df.groupBy(df.columns).count().filter(col('count') > 1).orderBy(col("user_id")).display()

In [0]:
from pyspark.sql.window import Window
df = df.withColumn('rn', row_number().over(Window.partitionBy(col('user_id')).orderBy(col('signup_date')))).filter(col('rn') == 1).drop('rn').display()

In [0]:
df.display()